In [1]:
import pandas as pd
import numpy as np
import re
import os
import requests
import time

In [ ]:
API_KEY = ""  

In [3]:
PATH = r'D:\hsenespr\project_hse_nursing_home\parsers'
files_needed = ['2gis_pansionaty_with_reviews.csv', 'yandex_pansionaty.csv', 'book_pansionaty.csv']
files_found = []

for f in files_needed:
    full_path = os.path.join(PATH, f)
    if os.path.exists(full_path):
        files_found.append(f)

In [4]:
df_2gis = pd.read_csv(os.path.join(PATH, '2gis_pansionaty_with_reviews.csv'), encoding='utf-8-sig')
df_2gis['source'] = '2GIS'
df_2gis['name'] = df_2gis['name']
df_2gis['address'] = df_2gis['address']
df_2gis['rating'] = pd.to_numeric(df_2gis['rating'], errors='coerce')
df_2gis['reviews'] = pd.to_numeric(df_2gis['reviews'], errors='coerce')
df_2gis['positive_words'] = pd.to_numeric(df_2gis['положительных_слов'], errors='coerce')
df_2gis['negative_words'] = pd.to_numeric(df_2gis['отрицательных_слов'], errors='coerce')
df_2gis['prices'] = df_2gis['цены'].notna() & (df_2gis['цены'] != '')
df_2gis['lat'] = df_2gis['lat']
df_2gis['lon'] = df_2gis['lon']

df_yandex = pd.read_csv(os.path.join(PATH, 'yandex_pansionaty.csv'), encoding='utf-8-sig')
df_yandex['source'] = 'Yandex'
df_yandex['name'] = df_yandex['название']
df_yandex['address'] = df_yandex['адрес']
df_yandex['rating'] = pd.to_numeric(df_yandex['рейтинг'], errors='coerce')
df_yandex['reviews'] = pd.to_numeric(df_yandex['число отзывов'], errors='coerce')
df_yandex['positive_words'] = pd.to_numeric(df_yandex['положительных_слов'], errors='coerce')
df_yandex['negative_words'] = pd.to_numeric(df_yandex['отрицательных_слов'], errors='coerce')
df_yandex['prices'] = df_yandex['цены'].notna() & (df_yandex['цены'] != '')
df_yandex['lat'] = None
df_yandex['lon'] = None

df_book = pd.read_csv(os.path.join(PATH, 'book_pansionaty.csv'), encoding='utf-8-sig')
df_book['source'] = 'Bookpansion'
df_book['name'] = df_book['title']
df_book['address'] = ''
df_book['rating'] = pd.to_numeric(df_book['rating'], errors='coerce')
df_book['reviews'] = pd.to_numeric(df_book['reviews'], errors='coerce')
df_book['positive_words'] = None
df_book['negative_words'] = None
df_book['prices'] = True
df_book['lat'] = None
df_book['lon'] = None

In [5]:
def extract_median_price_2gis(price_str):
    if pd.isna(price_str) or price_str == '':
        return None
    prices = re.findall(r'(\d+[\s]?\d*)\s*₽', str(price_str))
    if prices:
        nums = [int(p.replace(' ', '')) for p in prices]
        return int(np.median(nums))
    return None

def extract_median_price_yandex(price_str):
    if pd.isna(price_str) or price_str == '':
        return None
    prices = re.findall(r':\s*(\d+[\s]?\d*)', str(price_str))
    if prices:
        nums = [int(p.replace(' ', '')) for p in prices]
        return int(np.median(nums))
    return None

def extract_services_2gis(price_str):
    if pd.isna(price_str) or price_str == '':
        return ''
    services = []
    parts = str(price_str).split('|')
    for part in parts:
        if ':' in part:
            service = part.split(':')[0].strip()
            if service and service not in services:
                services.append(service)
    return ' | '.join(services)

def extract_services_yandex(price_str):
    if pd.isna(price_str) or price_str == '':
        return ''
    services = []
    parts = str(price_str).split(';')
    for part in parts:
        if ':' in part:
            service = part.split(':')[0].strip()
            if service and service not in services:
                services.append(service)
    return ' | '.join(services)




In [6]:
def get_coordinates_2gis(address):
    if pd.isna(address) or address == '':
        return None, None
    url = "https://catalog.api.2gis.com/3.0/items/geocode"
    params = {
        'q': address,
        'fields': 'items.point',
        'key': API_KEY
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        if data.get('result', {}).get('items'):
            point = data['result']['items'][0].get('point', {})
            return point.get('lat'), point.get('lon')
    except Exception as e:
        print(f"   Ошибка: {e}")
    
    return None, None

In [7]:
df_2gis['services'] = df_2gis['цены'].apply(extract_services_2gis)
df_yandex['services'] = df_yandex['цены'].apply(extract_services_yandex)
df_book['services'] = df_book['services'].fillna('') 

df_2gis['price'] = df_2gis['цены'].apply(extract_median_price_2gis)
df_yandex['price'] = df_yandex['цены'].apply(extract_median_price_yandex)
df_book['price'] = pd.to_numeric(df_book['min_price_per_day'], errors='coerce')

In [8]:
yandex_with_addr = df_yandex[df_yandex['address'].notna() & (df_yandex['address'] != '')]

for idx, row in yandex_with_addr.iterrows():
    address = row['address']
    print(f"      Геокодирование: {address}...")
    lat, lon = get_coordinates_2gis(address)
    df_yandex.loc[idx, 'lat'] = lat
    df_yandex.loc[idx, 'lon'] = lon
    time.sleep(0.3)

      Геокодирование: 1-я Радужная ул., 17, село Сельцо...
      Геокодирование: Бронницы, коттеджный посёлок Бронницы, 40...
      Геокодирование: Парковая ул., 7, д. Мишуткино...
      Геокодирование: Планерная ул., 35, микрорайон Авиационный, Домодедово...
      Геокодирование: д. Картино, 6В...
      Геокодирование: д. Дудкино, 35...
      Геокодирование: Садовая ул., 35А, ТЛПХ Подлипки, д. Малое Видное...
      Геокодирование: д. Ащерино, 2/4, Ворота открываются по звонку...
      Геокодирование: Серпуховская ул., 1Б, микрорайон Климовск, Подольск...
      Геокодирование: СНТ Ручеёк, 653...
      Геокодирование: Домодедово, территория Еленино, соор1/1...
      Геокодирование: посёлок Мещерино, СНТ Диалог, 10...
      Геокодирование: д. Глазынино, 35...
      Геокодирование: СНТ Гавриково, 58, Ворота открываются по звонку...
      Геокодирование: Профсоюзная ул., 140, корп. 6...
      Геокодирование: Лесная ул., 52, корп. 7А...
      Геокодирование: ул. Константинова, 42, п. г. т. 

In [10]:
columns = ['name', 'address', 'source', 'rating', 'reviews', 'positive_words', 'negative_words', 'prices', 'price', 'services', 'lat', 'lon']
df_combined = pd.concat([df_2gis[columns], df_yandex[columns], df_book[columns]], ignore_index=True)

df_combined['reviews'] = df_combined['reviews'].fillna(0)
df_combined['rating'] = df_combined['rating'].fillna(0)
df_combined = df_combined.sort_values('reviews', ascending=False)

df_with_address = df_combined[df_combined['address'].notna() & (df_combined['address'] != '')]
df_without_address = df_combined[df_combined['address'].isna() | (df_combined['address'] == '')]
df_with_address = df_with_address.drop_duplicates(subset=['address'], keep='first')
df_combined = pd.concat([df_with_address, df_without_address], ignore_index=True)
df_combined['services'] = df_combined['services'].fillna('')

MOSCOW_LAT_MIN = 54.2
MOSCOW_LAT_MAX = 56.8
MOSCOW_LON_MIN = 35.8
MOSCOW_LON_MAX = 39.8

invalid_coords_mask = (
    (df_combined['lat'] < MOSCOW_LAT_MIN) | 
    (df_combined['lat'] > MOSCOW_LAT_MAX) |
    (df_combined['lon'] < MOSCOW_LON_MIN) | 
    (df_combined['lon'] > MOSCOW_LON_MAX)
)
df_combined.loc[invalid_coords_mask, 'lat'] = None
df_combined.loc[invalid_coords_mask, 'lon'] = None

invalid_price_mask = df_combined['price'] > 15000
df_combined.loc[invalid_price_mask, 'price'] = None

invalid_reviews_mask = (df_combined['reviews'] > 0) & (df_combined['rating'] == 0)
df_combined.loc[invalid_reviews_mask, 'reviews'] = 0

df_combined.to_csv('all_pansionaty_combined.csv', index=False, encoding='utf-8-sig')